# Exercise

We are going to download CSV data from NOAA's website that gathers tornado data for both Texas and Oklahoma. We will then clean the data, select only the fields we are interested in, and load it into a SQLite database.

**STEP 1:** First import the necessary libraries. 

In [21]:
import pandas as pd
import sqlite3

**STEP 2:** Import data from CSV

In [22]:
df = pd.read_csv("storm_data.csv")
df

,EVENT_ID,CZ_NAME_STR,BEGIN_LOCATION,BEGIN_DATE,BEGIN_TIME,EVENT_TYPE,MAGNITUDE,TOR_F_SCALE,DEATHS_DIRECT,INJURIES_DIRECT,...,END_LOCATION,END_DATE,END_TIME,BEGIN_LAT,BEGIN_LON,END_LAT,END_LON,EVENT_NARRATIVE,EPISODE_NARRATIVE,STATE
0,1299990,OSAGE CO.,WYNONA,01/08/2026,841,Tornado,NaN,EF1,0,0,...,WYNONA,01/08/2026,842,36.5380,-96.3270,36.5470,-96.3180,"A tornado damaged the roofs of several homes, ...",A strong upper level disturbance and associate...,OKLAHOMA
1,1309165,LIBERTY CO.,KEVIN,02/03/2026,1524,Tornado,NaN,EF0,0,0,...,KEVIN,02/03/2026,1525,30.2531,-95.1021,30.2541,-95.0980,Storms moved through Liberty county on Februar...,"On the third of February, a tornado formed in ...",TEXAS
2,1309177,LIBERTY CO.,KEVIN,02/03/2026,1527,Tornado,NaN,EF0,0,0,...,PLUM GROVE,02/03/2026,1533,30.2547,-95.0866,30.2575,-95.0584,Storms moved through Liberty county on Februar...,"On the third of February, a tornado formed in ...",TEXAS
3,1311609,MCCLAIN CO.,PURCELL,01/08/2026,724,Tornado,NaN,EF2,0,0,...,PURCELL,01/08/2026,731,34.9670,-97.4430,35.0250,-97.3540,This tornado developed near 180th Street and L...,A fast-moving quasi-linear convective system (...,OKLAHOMA
4,1311610,CLEVELAND CO.,LEXINGTON,01/08/2026,731,Tornado,NaN,EF0,0,0,...,LEXINGTON,01/08/2026,734,35.0250,-97.3540,35.0470,-97.3110,This is a continuation of the Purcell tornado ...,A fast-moving quasi-linear convective system (...,OKLAHOMA
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
106,1339920,PUSHMATAHA CO.,ANTLERS,05/27/2026,1407,Tornado,NaN,EF1,0,0,...,ANTLERS,05/27/2026,1408,34.2306,-95.6147,34.2414,-95.6171,"This tornado damaged the hospital, damaged a f...",Thunderstorms increased across eastern Oklahom...,OKLAHOMA
107,1339922,CHOCTAW CO.,GOODLAND,05/27/2026,1745,Tornado,NaN,EFU,0,0,...,GOODLAND,05/27/2026,1745,33.9935,-95.5518,33.9935,-95.5518,This brief tornado was observed by several peo...,Thunderstorms increased across eastern Oklahom...,OKLAHOMA
108,1340662,GUADALUPE CO.,NEW BERLIN,05/26/2026,920,Tornado,NaN,EF1,0,0,...,NEW BERLIN,05/26/2026,925,29.5052,-98.0991,29.5029,-98.0508,The National Weather Service conducted a damag...,An upper shortwave trough moved across Texas a...,TEXAS
109,1340663,MAVERICK CO.,QUEMADO,05/26/2026,1619,Tornado,NaN,EF0,0,0,...,QUEMADO,05/26/2026,1622,29.0314,-100.6361,29.0243,-100.6335,The National Weather Service received reports ...,An upper shortwave trough moved across Texas a...,TEXAS


**STEP 3:** Extract out only the fields of interest.

In [23]:
fields = ["CZ_NAME_STR","BEGIN_LOCATION","BEGIN_DATE","BEGIN_TIME","TOR_F_SCALE",
          "DEATHS_DIRECT","INJURIES_DIRECT","DAMAGE_PROPERTY_NUM","DAMAGE_CROPS_NUM",
          "STATE_ABBR","END_LOCATION","END_DATE","END_TIME",
          "EVENT_NARRATIVE","EPISODE_NARRATIVE"]

df.drop(columns=[col for col in df if col not in fields], inplace=True)

df


,CZ_NAME_STR,BEGIN_LOCATION,BEGIN_DATE,BEGIN_TIME,TOR_F_SCALE,DEATHS_DIRECT,INJURIES_DIRECT,DAMAGE_PROPERTY_NUM,DAMAGE_CROPS_NUM,END_LOCATION,END_DATE,END_TIME,EVENT_NARRATIVE,EPISODE_NARRATIVE
0,OSAGE CO.,WYNONA,01/08/2026,841,EF1,0,0,200000,0,WYNONA,01/08/2026,842,"A tornado damaged the roofs of several homes, ...",A strong upper level disturbance and associate...
1,LIBERTY CO.,KEVIN,02/03/2026,1524,EF0,0,0,6000,0,KEVIN,02/03/2026,1525,Storms moved through Liberty county on Februar...,"On the third of February, a tornado formed in ..."
2,LIBERTY CO.,KEVIN,02/03/2026,1527,EF0,0,0,23000,0,PLUM GROVE,02/03/2026,1533,Storms moved through Liberty county on Februar...,"On the third of February, a tornado formed in ..."
3,MCCLAIN CO.,PURCELL,01/08/2026,724,EF2,0,0,400000,0,PURCELL,01/08/2026,731,This tornado developed near 180th Street and L...,A fast-moving quasi-linear convective system (...
4,CLEVELAND CO.,LEXINGTON,01/08/2026,731,EF0,0,0,10000,0,LEXINGTON,01/08/2026,734,This is a continuation of the Purcell tornado ...,A fast-moving quasi-linear convective system (...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
106,PUSHMATAHA CO.,ANTLERS,05/27/2026,1407,EF1,0,0,50000,0,ANTLERS,05/27/2026,1408,"This tornado damaged the hospital, damaged a f...",Thunderstorms increased across eastern Oklahom...
107,CHOCTAW CO.,GOODLAND,05/27/2026,1745,EFU,0,0,0,0,GOODLAND,05/27/2026,1745,This brief tornado was observed by several peo...,Thunderstorms increased across eastern Oklahom...
108,GUADALUPE CO.,NEW BERLIN,05/26/2026,920,EF1,0,0,0,0,NEW BERLIN,05/26/2026,925,The National Weather Service conducted a damag...,An upper shortwave trough moved across Texas a...
109,MAVERICK CO.,QUEMADO,05/26/2026,1619,EF0,0,0,0,0,QUEMADO,05/26/2026,1622,The National Weather Service received reports ...,An upper shortwave trough moved across Texas a...


**STEP 4:** Convert date/time fields to a single datetime in new fields. Clean up the times so they have 4 digits and a colon. Then Convert those new fields to UTC. Finally, drop the original date/time fields.

In [24]:
def clean_time(time):
    time_str = str(time).strip()
    c = f"{'0' * (4-len(time_str))}{time_str}"
    return c[0:2] + ":" + c[2:4]

df.insert(2, 'BEGIN_DATETIME', pd.to_datetime(df['BEGIN_DATE'] + ' ' + df['BEGIN_TIME'].apply(clean_time))  \
    .dt.tz_localize('US/Central') \
    .dt.tz_convert('UTC')
          )

df.insert(3, 'END_DATETIME', pd.to_datetime(df['END_DATE'] + ' ' + df['END_TIME'].apply(clean_time))  \
    .dt.tz_localize('US/Central') \
    .dt.tz_convert('UTC')
)

df.drop(["BEGIN_DATE", "BEGIN_TIME", "END_DATE", "END_TIME"], axis=1, inplace=True)

df

,CZ_NAME_STR,BEGIN_LOCATION,BEGIN_DATETIME,END_DATETIME,TOR_F_SCALE,DEATHS_DIRECT,INJURIES_DIRECT,DAMAGE_PROPERTY_NUM,DAMAGE_CROPS_NUM,END_LOCATION,EVENT_NARRATIVE,EPISODE_NARRATIVE
0,OSAGE CO.,WYNONA,2026-01-08 14:41:00+00:00,2026-01-08 14:42:00+00:00,EF1,0,0,200000,0,WYNONA,"A tornado damaged the roofs of several homes, ...",A strong upper level disturbance and associate...
1,LIBERTY CO.,KEVIN,2026-02-03 21:24:00+00:00,2026-02-03 21:25:00+00:00,EF0,0,0,6000,0,KEVIN,Storms moved through Liberty county on Februar...,"On the third of February, a tornado formed in ..."
2,LIBERTY CO.,KEVIN,2026-02-03 21:27:00+00:00,2026-02-03 21:33:00+00:00,EF0,0,0,23000,0,PLUM GROVE,Storms moved through Liberty county on Februar...,"On the third of February, a tornado formed in ..."
3,MCCLAIN CO.,PURCELL,2026-01-08 13:24:00+00:00,2026-01-08 13:31:00+00:00,EF2,0,0,400000,0,PURCELL,This tornado developed near 180th Street and L...,A fast-moving quasi-linear convective system (...
4,CLEVELAND CO.,LEXINGTON,2026-01-08 13:31:00+00:00,2026-01-08 13:34:00+00:00,EF0,0,0,10000,0,LEXINGTON,This is a continuation of the Purcell tornado ...,A fast-moving quasi-linear convective system (...
...,...,...,...,...,...,...,...,...,...,...,...,...
106,PUSHMATAHA CO.,ANTLERS,2026-05-27 19:07:00+00:00,2026-05-27 19:08:00+00:00,EF1,0,0,50000,0,ANTLERS,"This tornado damaged the hospital, damaged a f...",Thunderstorms increased across eastern Oklahom...
107,CHOCTAW CO.,GOODLAND,2026-05-27 22:45:00+00:00,2026-05-27 22:45:00+00:00,EFU,0,0,0,0,GOODLAND,This brief tornado was observed by several peo...,Thunderstorms increased across eastern Oklahom...
108,GUADALUPE CO.,NEW BERLIN,2026-05-26 14:20:00+00:00,2026-05-26 14:25:00+00:00,EF1,0,0,0,0,NEW BERLIN,The National Weather Service conducted a damag...,An upper shortwave trough moved across Texas a...
109,MAVERICK CO.,QUEMADO,2026-05-26 21:19:00+00:00,2026-05-26 21:22:00+00:00,EF0,0,0,0,0,QUEMADO,The National Weather Service received reports ...,An upper shortwave trough moved across Texas a...


**STEP 5:** Rename a fiew fields to make them easier to identify for end users.

In [25]:
df.rename(columns= { 
    "CZ_NAME_STR": "COUNTY_NAME",
    "DAMAGE_PROPERTY_NUM" :"DAMAGE_PROPERTY_USD", 
    "DAMAGE_CROPS_NUM" :"DAMAGE_CROPS_USD"
})


,COUNTY_NAME,BEGIN_LOCATION,BEGIN_DATETIME,END_DATETIME,TOR_F_SCALE,DEATHS_DIRECT,INJURIES_DIRECT,DAMAGE_PROPERTY_USD,DAMAGE_CROPS_USD,END_LOCATION,EVENT_NARRATIVE,EPISODE_NARRATIVE
0,OSAGE CO.,WYNONA,2026-01-08 14:41:00+00:00,2026-01-08 14:42:00+00:00,EF1,0,0,200000,0,WYNONA,"A tornado damaged the roofs of several homes, ...",A strong upper level disturbance and associate...
1,LIBERTY CO.,KEVIN,2026-02-03 21:24:00+00:00,2026-02-03 21:25:00+00:00,EF0,0,0,6000,0,KEVIN,Storms moved through Liberty county on Februar...,"On the third of February, a tornado formed in ..."
2,LIBERTY CO.,KEVIN,2026-02-03 21:27:00+00:00,2026-02-03 21:33:00+00:00,EF0,0,0,23000,0,PLUM GROVE,Storms moved through Liberty county on Februar...,"On the third of February, a tornado formed in ..."
3,MCCLAIN CO.,PURCELL,2026-01-08 13:24:00+00:00,2026-01-08 13:31:00+00:00,EF2,0,0,400000,0,PURCELL,This tornado developed near 180th Street and L...,A fast-moving quasi-linear convective system (...
4,CLEVELAND CO.,LEXINGTON,2026-01-08 13:31:00+00:00,2026-01-08 13:34:00+00:00,EF0,0,0,10000,0,LEXINGTON,This is a continuation of the Purcell tornado ...,A fast-moving quasi-linear convective system (...
...,...,...,...,...,...,...,...,...,...,...,...,...
106,PUSHMATAHA CO.,ANTLERS,2026-05-27 19:07:00+00:00,2026-05-27 19:08:00+00:00,EF1,0,0,50000,0,ANTLERS,"This tornado damaged the hospital, damaged a f...",Thunderstorms increased across eastern Oklahom...
107,CHOCTAW CO.,GOODLAND,2026-05-27 22:45:00+00:00,2026-05-27 22:45:00+00:00,EFU,0,0,0,0,GOODLAND,This brief tornado was observed by several peo...,Thunderstorms increased across eastern Oklahom...
108,GUADALUPE CO.,NEW BERLIN,2026-05-26 14:20:00+00:00,2026-05-26 14:25:00+00:00,EF1,0,0,0,0,NEW BERLIN,The National Weather Service conducted a damag...,An upper shortwave trough moved across Texas a...
109,MAVERICK CO.,QUEMADO,2026-05-26 21:19:00+00:00,2026-05-26 21:22:00+00:00,EF0,0,0,0,0,QUEMADO,The National Weather Service received reports ...,An upper shortwave trough moved across Texas a...


**STEP 6:** Load the data into a SQLite database file, into a table called `TORNADO_TRACK`.

In [26]:
conn = sqlite3.connect('my_database.db')
df.to_sql("TORNADO_TRACK", conn, if_exists='replace', index=False)

# 4. VERIFY DATA IS LOADED USING A SELECT query 
sql_df = pd.read_sql("SELECT * FROM TORNADO_TRACK", conn)
with pd.option_context('display.max_rows', None, 'display.max_colwidth', None):
  display(sql_df)

conn.close()

,CZ_NAME_STR,BEGIN_LOCATION,BEGIN_DATETIME,END_DATETIME,TOR_F_SCALE,DEATHS_DIRECT,INJURIES_DIRECT,DAMAGE_PROPERTY_NUM,DAMAGE_CROPS_NUM,END_LOCATION,EVENT_NARRATIVE,EPISODE_NARRATIVE
0,OSAGE CO.,WYNONA,2026-01-08 14:41:00+00:00,2026-01-08 14:42:00+00:00,EF1,0,0,200000,0,WYNONA,"A tornado damaged the roofs of several homes, destroyed a large outbuilding, snapped power poles, and blew down trees. Based on this damage, maximum estimated wind in the tornado was 90 to 100 mph.","A strong upper level disturbance and associated cold front translated into the Southern Plains on the 8th. An unseasonably warm and moist air mass had spread into eastern Oklahoma ahead of this system, which resulted in weak instability across the area. A line of thunderstorms developed into eastern Oklahoma from the west during the morning hours of the 8th. Strong wind fields and wind shear associated with the approaching upper level system, combined with the weak instability, supported an environment in which the stronger thunderstorms within the squall line were able to develop low level circulations. One such circulation resulted in a brief, weak tornado and wind gusts to 88 mph in Osage County."
1,LIBERTY CO.,KEVIN,2026-02-03 21:24:00+00:00,2026-02-03 21:25:00+00:00,EF0,0,0,6000,0,KEVIN,"Storms moved through Liberty county on February 3rd, where a tornado warned storm passed through the Plum Grove area. Initially broadcast media reported trees down near intersection of County Road 3709 and County Road 3732. In the aftermath of this storm, a survey team was dispatched to assess the damage across the area.||The survey team found that this storm produced two tornadoes, this being the first. According to the survey team, the tornado appeared to initially touched down within a wooded area between CR 3708 and CR 3709, then turned eastward along CR 3709 for a short distance before lifting up shortly before River Ln. A drone was used to assess an areal view of the damage, though the survey team noted that they did not see any damage in the deeply wooded area along the East Fork of the San Jacinto River. Therefore, the survey team concluded that the first tornado ended just before the River Ln and CR 3709 intersection. The majority of the damage seen was broken trunks and branches of pine trees and fences down. Power lines were also reported blown down, but the power line poles remained upright. This tornado was rated as an EF0 with a maximum estimated wind speed of 80 mph. This maximum wind speed was based on Softwood Trees (TS) with a DOD of 2. Survey teams noted that the trees in question had reported beetle damage.","On the third of February, a tornado formed in Plum Grove area in Liberty county."
2,LIBERTY CO.,KEVIN,2026-02-03 21:27:00+00:00,2026-02-03 21:33:00+00:00,EF0,0,0,23000,0,PLUM GROVE,"Storms moved through Liberty county on February 3rd, where a tornado warned storm passed through the Plum Grove area. Initially broadcast media reported trees down near intersection of County Road 3709 and County Road 3732. In the aftermath of this storm, a survey team was dispatched to assess the damage across the area.||The survey team found that this storm produced two tornadoes, this being the second. This second tornado started along FM 1010 and Wooden Ln, indicated by an uprooted tree and broken wood signage. The tornado entered a largely empty field between FM 1010 and CR 3405, but still saw some tree damage. The tornado then entered the Bella Vista subdivision, causing tree damage, fence damage, and metal structure damage between CR 3405 and CR 3404. No further damage was seen beyond the intersection of CR 3404 and CR 3410. This tornado was rated as an EF0. The most significant damage occurred at a metal structure off of FM 1010 with estimated peak winds of 85 mph. This assessment is based on a metal building system (MBS) with a DOD of 3.","On the third of February, a tornado formed in Plum Grove area in Liberty county."
3,MCCLAIN CO.,PURCELL,2026-01-08 13:24